# Biological stage variability within cohorts

The reliability notebook (`morph_stage_reliability.ipynb`) asked *how trustworthy is each embryo's stage estimate* (epistemic / model uncertainty). **This notebook asks a different question**: how much does **true developmental stage** actually spread **within each (temperature, timepoint) cohort** — the biological desynchronization — after removing the model's estimation noise.

### Variance decomposition
Observed within-cohort spread mixes real biology with model wobble:
$$\mathrm{Var}(\text{observed stage}) \;\approx\; \underbrace{\mathrm{Var}_\text{bio}}_{\text{what we want}} \;+\; \underbrace{\overline{\mathrm{Var}_\text{est}}}_{\text{mean per-embryo estimation variance}}$$
so $\mathrm{Var}_\text{bio} \approx \mathrm{Var}_\text{observed} - \overline{\mathrm{Var}_\text{est}}$ (floored at 0). The per-embryo estimation variance comes from a single embryo-level bootstrap **refit ensemble** of the ref-only polynomial (the only place model refits happen; cached to disk).

### Three estimators, as a leakage cross-check (Nick's point)
- **`mdl_stage_hpf`** — pure surface (degree-3 poly). *Primary.* Its gradient is not guaranteed tangent to the developmental manifold, so off-support it *can* let orthogonal (abnormality) spread leak into stage.
- **`spline_stage_hpf`** — Euclidean projection to the WT spline. Orthogonal spread is **definitionally** in the null space → cohort SD is progression-only variability.
- **`nn_stage_hpf`** — nearest reference embryo (non-parametric).

Where surface ≈ projection SD, the surface is behaving like a projection (no leakage) and the biological number is clean; where surface > projection, leakage is flagged in that cohort.

### Decisions
Primary = surface. Outliers shown **gated & ungated side-by-side** (gate = polynomial leverage ≤ reference 99th pct). Spread shown **instantaneous** (headline) and **rate-normalized** (per hpf of development).

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import morph_stage_reliability_utils as mr

plt.style.use('default')
mpl.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
                     'font.size': 10, 'savefig.bbox': 'tight'})

N_BOOT = 300         # refit ensemble size (cached to disk)
TIMEPOINT_MARKERS = {24.0: 'o', 30.0: 's', 36.0: '^'}
print('data cache:', mr.CACHE_DIR)
print('figure dir:', mr.fig_dir())

## Setup: model, three estimators, support gate, refit ensemble

In [ ]:
ref, hf, spline = mr.load_tables()
hf = mr.attach_morph_dist_spline(hf)
train = mr.reference_training_frame(ref)
model = mr.build_model(train)

# three stage estimators on the hotfish points
hf['mdl_stage_hpf'] = model.predict(hf[mr.PCA_COLS].values)        # surface (primary)
hf['spline_stage_hpf'] = mr.compute_spline_stage(hf, spline)       # projection
# nn_stage_hpf is already in the cached table

# support gate: polynomial leverage <= reference 99th percentile
hf['leverage'] = mr.polynomial_leverage(model, train, hf)
lev99 = np.quantile(mr.polynomial_leverage(model, train, train), 0.99)
support_mask = (hf['leverage'] <= lev99).to_numpy()
print(f'support gate: {support_mask.sum()}/{len(hf)} embryos pass (leverage <= {lev99:.4f})')

# the single refit ensemble (cached ~5 min build, instant thereafter)
ens = mr.cached_refit_ensemble(train, hf, n_boot=N_BOOT)
print(f'refit ensemble: {ens.shape}')

## P1 — Biological stage variability vs temperature (headline)

Within-cohort stage SD from the surface estimator, decomposed into observed vs biological (estimation noise subtracted), shown **gated vs ungated** and split by collection timepoint. The gap between observed and biological is the model's own contribution; the gap between gated and ungated is what the off-support outliers add.

In [ ]:
var_all = mr.cohort_stage_variability(hf, ens, stage_col='mdl_stage_hpf', support_mask=None)
var_gated = mr.cohort_stage_variability(hf, ens, stage_col='mdl_stage_hpf', support_mask=support_mask)

timepoints = sorted(hf['timepoint'].unique())
fig, axes = plt.subplots(1, len(timepoints), figsize=(4.6 * len(timepoints), 4.2), sharey=True)
for ax, tp in zip(np.atleast_1d(axes), timepoints):
    for tab, color, lab, ls in [(var_all, '#999999', 'observed (all)', ':'),
                                (var_gated, '#b2182b', 'biological (gated)', '-'),
                                (var_all, '#2166ac', 'biological (all)', '--')]:
        d = tab[np.isclose(tab['timepoint'], tp)].sort_values('temperature')
        ycol = 'observed_sd' if 'observed' in lab else 'biological_sd'
        yerr = None if 'observed' in lab else d['biological_sd_se']
        ax.errorbar(d['temperature'], d[ycol], yerr=yerr, fmt='o' + ls if ls != ':' else 'o:',
                    color=color, ecolor=color, elinewidth=0.9, capsize=2.5,
                    markersize=5, lw=1.4, label=lab, alpha=0.9)
    ax.set_title(f'{tp:g} hpf collection')
    ax.set_xlabel('temperature (C)')
axes_flat = np.atleast_1d(axes)
axes_flat[0].set_ylabel('within-cohort stage SD (hpf)')
axes_flat[0].legend(frameon=False, fontsize=8, loc='upper left')
fig.suptitle('Biological stage variability grows with temperature (surface estimator)', y=1.02)
mr.savefig(fig, 'P1_biological_stage_sd_vs_temp')
plt.show()
var_gated.round(3)

## P2 — Three-estimator agreement (leakage cross-check)

Per-cohort SD from surface / projection / kNN. Where the **surface** SD (red) exceeds the **projection** SD (blue), off-manifold variation is leaking into the surface stage estimate for that cohort. Broad agreement confirms the surface behaves like a projection on-support, so the P1 biological number is trustworthy there.

In [ ]:
me = mr.multi_estimator_cohort_sd(
    hf, estimator_cols=['mdl_stage_hpf', 'spline_stage_hpf', 'nn_stage_hpf'])
me['cohort'] = me['temperature'].astype(str) + 'C/' + me['timepoint'].astype(int).astype(str) + 'h'
me = me.sort_values(['temperature', 'timepoint']).reset_index(drop=True)

x = np.arange(len(me)); w = 0.27
fig, ax = plt.subplots(figsize=(13, 4.4))
ax.bar(x - w, me['mdl_stage_hpf_sd'], w, color='#b2182b', label='surface (mdl)')
ax.bar(x, me['spline_stage_hpf_sd'], w, color='#2166ac', label='projection (spline)')
ax.bar(x + w, me['nn_stage_hpf_sd'], w, color='#999999', label='kNN (nn)')
ax.set_xticks(x); ax.set_xticklabels(me['cohort'], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('within-cohort stage SD (hpf)')
ax.set_title('Stage-SD by estimator per cohort — surface vs projection flags abnormality leakage')
ax.legend(frameon=False, fontsize=9)
mr.savefig(fig, 'P2_three_estimator_agreement')
plt.show()

me['surface_minus_projection'] = me['mdl_stage_hpf_sd'] - me['spline_stage_hpf_sd']
print('Cohorts where surface SD most exceeds projection SD (candidate leakage):')
me.sort_values('surface_minus_projection', ascending=False)[
    ['cohort', 'n', 'mdl_stage_hpf_sd', 'spline_stage_hpf_sd', 'surface_minus_projection']].head(6).round(3)

## P3 — Variance decomposition (what fraction of cohort spread is real?)

Per cohort, the observed **variance** split into biological + estimation components (gated). Where the estimation bar dominates, the raw cohort SD was mostly model noise — a warning that the observed spread there is not a clean biological readout.

In [ ]:
vg = var_gated.copy()
vg['cohort'] = vg['temperature'].astype(str) + 'C/' + vg['timepoint'].astype(int).astype(str) + 'h'
vg = vg.sort_values(['temperature', 'timepoint']).reset_index(drop=True)
vg['bio_var'] = vg['biological_sd'] ** 2
vg['est_var'] = vg['estimation_sd'] ** 2

x = np.arange(len(vg))
fig, ax = plt.subplots(figsize=(13, 4.4))
ax.bar(x, vg['bio_var'], color='#4daf4a', label='biological variance')
ax.bar(x, vg['est_var'], bottom=vg['bio_var'], color='#d9a441', label='estimation variance')
ax.set_xticks(x); ax.set_xticklabels(vg['cohort'], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('within-cohort stage variance (hpf$^2$)')
ax.set_title('Variance decomposition per cohort (gated): biological vs estimation')
ax.legend(frameon=False, fontsize=9)
mr.savefig(fig, 'P3_variance_decomposition')
plt.show()

## P4 — Rate-normalized spread (secondary)

Instantaneous SD conflates 'already spread' with 'fanning out fast': a 36 hpf cohort has had more developmental time for rate differences to accumulate than a 24 hpf one. Normalizing the biological SD by elapsed development (nominal stage) expresses spread as a **fraction of development elapsed** — a fan-out rate, comparable across timepoints.

In [ ]:
vg2 = var_gated.copy()
# elapsed development proxy = nominal collection stage (hpf since fertilization)
vg2['bio_sd_per_hpf'] = vg2['biological_sd'] / vg2['timepoint']
vg2['bio_sd_per_hpf_se'] = vg2['biological_sd_se'] / vg2['timepoint']

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, (ycol, yerr, ylab, title) in zip(axes, [
        ('biological_sd', 'biological_sd_se', 'biological stage SD (hpf)', 'instantaneous'),
        ('bio_sd_per_hpf', 'bio_sd_per_hpf_se', 'biological SD / hpf elapsed', 'rate-normalized')]):
    for tp in sorted(vg2['timepoint'].unique()):
        d = vg2[np.isclose(vg2['timepoint'], tp)].sort_values('temperature')
        ax.errorbar(d['temperature'], d[ycol], yerr=d[yerr],
                    fmt=TIMEPOINT_MARKERS.get(tp, 'o') + '-', markersize=6, capsize=2.5,
                    lw=1.4, label=f'{tp:g} hpf')
    ax.set_xlabel('temperature (C)'); ax.set_ylabel(ylab); ax.set_title(title)
axes[0].legend(frameon=False, fontsize=8, title='collection')
fig.suptitle('Biological stage spread: instantaneous vs rate-normalized (gated)', y=1.02)
mr.savefig(fig, 'P4_rate_normalized_spread')
plt.show()

### Takeaways (first pass)

- **P1**: after removing model estimation noise, biological stage SD rises with temperature — cold/control cohorts ~0.5–0.9 hpf, hot cohorts several hpf — consistent with heat-driven developmental desynchronization. Gated vs ungated shows how much of the raw hot-cohort spread was off-support outliers vs real spread.
- **P2**: surface and projection SD broadly agree, so the surface behaves like a projection on-support (no abnormality leakage); cohorts where surface > projection are flagged.
- **P3**: warns which cohorts' raw spread was dominated by estimation noise (subtraction unstable) — those biological numbers are least reliable.
- **P4**: rate-normalized view separates 'already spread' from 'fast fan-out'.

**Iterate from here**: per-embryo (trajectory) rather than per-frame spread; propagate biological SD into the downstream cohort-mean tempo figures; formal test of SD-vs-temperature trend.